In [1]:
%config Completer.use_jedi = False

In [2]:
# load module
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import copy
import warnings
import torch
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorboard as tb 
import pytorch_lightning as pl
from itertools import product

from pathlib import Path
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet , EncoderNormalizer , GroupNormalizer
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
#from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
from IPython.core.display import HTML

import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")  # avoid printing out absolute paths
plt.rcParams['font.family'] = 'NanumGothic'
#plt.rcParams['font.sans-serif'] = ['NanumGothic.ttf', 'sans-serif']

from my_funs import *

/home/nplab/.local/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: /home/nplab/.local/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIlEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  warn(f"Failed to load image Python extension: {e}")
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will n

In [3]:
funs = ['com' , 'span' , 'halflife']
fun = [moving_average_alpha, moving_average_com, moving_average_span, moving_average_halflife]
result_df1 = pd.DataFrame(columns = ['funs' , 'factor'])
result_df2 = pd.DataFrame(columns = ['funs' , 'factor'])

r1 = np.arange(0,10)/10
result_df1 = pd.DataFrame(product(['alpha'], r1))
result_df1.columns = ['funs' , 'factor']


r2 = np.arange(1 , 15 , 3)
result_df2 = pd.DataFrame(product(funs, r2))
result_df2.columns = ['funs' , 'factor']

result_df = pd.concat([result_df1,result_df2 ])
result_df['fun'] = 0

result_df.reset_index(inplace=True, drop=True)


idx = result_df[result_df['funs'] == 'alpha'].index
result_df.loc[idx, 'fun'] = moving_average_alpha

idx = result_df[result_df['funs'] == 'com'].index
result_df.loc[idx, 'fun'] = moving_average_com

idx = result_df[result_df['funs'] == 'span'].index
result_df.loc[idx, 'fun'] = moving_average_span

idx = result_df[result_df['funs'] == 'halflife'].index
result_df.loc[idx, 'fun'] = moving_average_halflife


result_df

,funs,factor,fun
0,alpha,0.0,<function moving_average_alpha at 0x7fe716d28d30>
1,alpha,0.1,<function moving_average_alpha at 0x7fe716d28d30>
2,alpha,0.2,<function moving_average_alpha at 0x7fe716d28d30>
3,alpha,0.3,<function moving_average_alpha at 0x7fe716d28d30>
4,alpha,0.4,<function moving_average_alpha at 0x7fe716d28d30>
5,alpha,0.5,<function moving_average_alpha at 0x7fe716d28d30>
6,alpha,0.6,<function moving_average_alpha at 0x7fe716d28d30>
7,alpha,0.7,<function moving_average_alpha at 0x7fe716d28d30>
8,alpha,0.8,<function moving_average_alpha at 0x7fe716d28d30>
9,alpha,0.9,<function moving_average_alpha at 0x7fe716d28d30>


In [4]:
epoch = 100
hidden_size = 16
atten_head = 8
batch_size= 16
pl.seed_everything(42)

for idx in result_df.index:
    data_path = '../../train.csv'
    fun = result_df['fun'].loc[idx]
    factor = result_df['factor'].loc[idx]
    print(idx, fun , factor)
    data = data_processing(data_path, factor, fun)
    
    train_data = data[data['time_idx']  < 8760]
    training = get_training(train_data,24, 24*7)

    dataloader = training.to_dataloader(batch_size)
    validation = TimeSeriesDataSet.from_dataset(training, train_data, predict=True, stop_randomization=True)
    train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
    val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size*100, num_workers=0)

    early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min")
    lr_logger = LearningRateMonitor()  
    logger = TensorBoardLogger('ewma_test')
    trainer = pl.Trainer(
        max_epochs=300,
        gpus=1,
        gradient_clip_val=0.1,
        limit_train_batches=20,  
        callbacks=[lr_logger],
        logger=logger,
    )


    tft = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=0.01,
        hidden_size=hidden_size,  # most important hyperparameter apart from learning rate
        # number of attention heads. Set to up to 4 for large datasets
        attention_head_size=atten_head,
        dropout=0.1,  # between 0.1 and 0.3 are good values
        hidden_continuous_size=hidden_size,  # set to <= hidden_size
        output_size=7,  # 7 quantiles by default
        loss=QuantileLoss(),
        # reduce learning rate if no improvement in validation loss after x epochs
        reduce_on_plateau_patience=4,
    )

    tft.hparams['optimizer'] = 'ranger'

    trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
    )
    
    print(trainer.checkpoint_callback.best_model_path)
    result_df.loc[idx,'model_ckpt'] = trainer.checkpoint_callback.best_model_path
    result_df.to_csv('ewma_setting.csv')


Global seed set to 42


0 <function moving_average_alpha at 0x7fe716d28d30> 0.0


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Missing logger folder: ewma_test/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 538   
3  | prescalers                         | ModuleDict                      | 224   
4  | static_variable_selection          | VariableSelectionNetwork        | 1.3 K 
5  | encoder_variable_selection         | VariableSelectionNetwork        | 9.1 K 
6  | decoder_variable_selection         | VariableSelectionNetwork        | 3.1 K 
7  |

Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=300` reached.


ewma_test/lightning_logs/version_24/checkpoints/epoch=299-step=6000.ckpt
